In [1]:
!pip install pandas scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv("raw_skills.csv")
df["skills_text"] = df["skills"].apply(
    lambda s: " ".join(skill.strip().replace(" ", "_") for skill in s.split(","))
)
df

,job_role,skills,skills_text
0,Data Scientist,"Python, SQL, Machine Learning, Statistics, Dat...",Python SQL Machine_Learning Statistics Data_An...
1,DevOps Engineer,"AWS, Docker, Kubernetes, CI/CD, Automation, Li...",AWS Docker Kubernetes CI/CD Automation Linux C...
2,Backend Developer,"Java, Python, SQL, APIs, REST, Spring Boot, Da...",Java Python SQL APIs REST Spring_Boot Database...
3,Frontend Developer,"HTML, CSS, JavaScript, React, UI Design, Web D...",HTML CSS JavaScript React UI_Design Web_Design...
4,Cloud Architect,"AWS, Azure, Cloud, Automation, Networking, Sec...",AWS Azure Cloud Automation Networking Security...
5,System Administrator,"Linux, Networking, Automation, Bash, Security,...",Linux Networking Automation Bash Security Clou...
6,Machine Learning Engineer,"Python, Machine Learning, TensorFlow, Deep Lea...",Python Machine_Learning TensorFlow Deep_Learni...
7,Full Stack Developer,"JavaScript, React, Node.js, SQL, APIs, Git, HT...",JavaScript React Node.js SQL APIs Git HTML CSS
8,Data Engineer,"Python, SQL, ETL, Data Structures, Cloud, AWS,...",Python SQL ETL Data_Structures Cloud AWS Spark...
9,Security Engineer,"Networking, Security, Linux, Automation, Cloud...",Networking Security Linux Automation Cloud Enc...


In [4]:
def ingest_user_skills(user_skills):
    if len(user_skills) < 3:
        raise ValueError("Kam se kam 3 skills do")
    return " ".join(skill.strip().replace(" ", "_") for skill in user_skills)

def score_roles(df, user_text):
    corpus = df["skills_text"].tolist() + [user_text]
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus)
    user_vector = tfidf_matrix[-1]
    role_vectors = tfidf_matrix[:-1]
    scores = cosine_similarity(user_vector, role_vectors).flatten()
    df = df.copy()
    df["similarity_score"] = scores
    return df

def get_top_n_recommendations(df, n=3):
    sorted_df = df.sort_values(by="similarity_score", ascending=False)
    return sorted_df.head(n)[["job_role", "similarity_score", "skills"]]